<a href="https://colab.research.google.com/github/markjschmidt/PET_List/blob/main/MJS_Sonnet4_PET_MIP_S46712_20261018.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## PET Maximum Intensity Projection — Rotating MIP Viewer

Loads DICOM PET slices from GitHub (via Git LFS), builds a 3D volume, pre-computes rotating MIP projections, and displays an interactive viewer with angle and colormap controls.

**Study:** S46712 — 128 slices (I10–I1280), ~125 KB/slice  
**Dataset path:** `DICOM/S46712/` (flat series — no sub-series folder)  
**Orientation:** Coronal — slices stack along **Y (anterior → posterior)**; each slice is a ZX plane (superior-inferior × left-right).  
> Units, pixel spacing and Y range are read from the DICOM headers after LFS pull (Step 3).

## Step 1 — Install dependencies and clone data

In [ ]:
# Install git-lfs (not present by default in Colab)
!apt-get install -y git-lfs -q

# Install Python packages
!pip install pydicom scipy ipywidgets -q

# Enable ipywidgets in Colab output
from google.colab import output
output.enable_custom_widget_manager()

## Step 2 — Clone repo and pull DICOM files from Git LFS

In [ ]:
import os

REPO_URL  = 'https://github.com/markjschmidt/PET_MIP'
CLONE_DIR = '/content/PET_MIP'
DICOM_DIR = f'{CLONE_DIR}/DICOM/S46712'

if not os.path.isdir(CLONE_DIR):
    !git clone --depth 1 {REPO_URL} {CLONE_DIR}
    !git -C {CLONE_DIR} lfs pull --include='DICOM/S46712/*'
else:
    print('Repo already cloned — skipping.')

# Verify
import glob
files = sorted(glob.glob(f'{DICOM_DIR}/I*'))
print(f'Found {len(files)} DICOM files.')

# Confirm LFS objects were fetched (pointer stubs are 131 bytes; real DICOM > 100 KB)
sizes = [os.path.getsize(f) for f in files[:3]]
if all(s < 500 for s in sizes):
    print('WARNING: Files look like LFS pointer stubs — LFS pull may have failed.')
    print('Run: !git -C {CLONE_DIR} lfs pull --include="DICOM/S46712/*"')
else:
    print(f'LFS pull confirmed — first slice size: {sizes[0]:,} bytes')

## Step 3 — Build 3D volume (coronal series)
Sorts slices by **Y position** (`ImagePositionPatient[1]`, anterior → posterior) and applies the DICOM rescale slope/intercept.  
Each slice is a ZX plane: rows = superior-inferior (Z), columns = left-right (X).

In [ ]:
import pydicom
import numpy as np

# ── Sort coronal slices by Y position (anterior → posterior) ─────────────
slice_list = []
for f in files:
    ds = pydicom.dcmread(f, stop_before_pixels=True)
    y  = float(ds.ImagePositionPatient[1])   # Y = A-P axis for coronal slices
    slice_list.append((y, f))
slice_list.sort(key=lambda x: x[0])

# ── Read metadata from first slice ───────────────────────────────────────
ds0         = pydicom.dcmread(slice_list[0][1], stop_before_pixels=True)
units       = getattr(ds0, 'Units',            'unknown')   # tag (0054,1001)
px_spacing  = getattr(ds0, 'PixelSpacing',      [2.0, 2.0]) # [row_spacing(Z), col_spacing(X)]
rows        = getattr(ds0, 'Rows',              None)
cols        = getattr(ds0, 'Columns',           None)
series_desc = getattr(ds0, 'SeriesDescription', 'S46712')

# Slice (Y) spacing from adjacent slice positions
y_spacing = abs(slice_list[1][0] - slice_list[0][0]) if len(slice_list) > 1 else float(px_spacing[0])
z_spacing = float(px_spacing[0])   # row spacing  = Z (sup-inf) within slice
x_spacing = float(px_spacing[1])   # col spacing  = X (L-R)     within slice

print(f'Series        : {series_desc}')
print(f'Units         : {units}    (tag 0054,1001)')
print(f'Matrix        : {rows} rows (Z) x {cols} cols (X)')
print(f'In-plane px   : {z_spacing} mm (Z) x {x_spacing} mm (X)')
print(f'Slice spacing : {y_spacing:.3f} mm (Y, A-P)')
print(f'Y range       : {slice_list[0][0]:.1f} → {slice_list[-1][0]:.1f} mm')

# ── Load pixel data and apply rescale ────────────────────────────────────
volume = []
for y, f in slice_list:
    ds     = pydicom.dcmread(f)
    arr    = ds.pixel_array.astype(np.float32)
    slope  = float(getattr(ds, 'RescaleSlope',     1))
    intcpt = float(getattr(ds, 'RescaleIntercept', 0))
    volume.append(arr * slope + intcpt)

# vol.shape = (N_slices=Y_dim, rows=Z_dim, cols=X_dim)
vol = np.array(volume)
print(f'\nVolume shape  : {vol.shape}  (Y, Z, X)')
print(f'Pixel range   : {vol.min():.1f} → {vol.max():.1f} {units}')

## Step 3b — Verify volume with interactive slice viewer

Scroll through individual **coronal slices** (the raw acquired planes) and **axial cross-sections** (Y–X cuts through the stacked volume) to confirm correct orientation and data quality before spending time on MIP computation.

- **Coronal slice** `vol[y, :, :]` — shape (Z, X): each acquired image  
- **Axial cross-section** `vol[:, z, :]` — shape (Y, X): horizontal cut at a given Z level

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import ipywidgets as widgets
from IPython.display import display

n_cor  = vol.shape[0]   # number of coronal slices (Y axis)
n_z    = vol.shape[1]   # rows per slice (Z axis, sup-inf)
n_x    = vol.shape[2]   # cols per slice (X axis, L-R)

# For a uniform phantom, 95th percentile avoids saturation better than 99.5th
SLICE_VMAX = float(np.percentile(vol[vol > 0], 95))

# ── Sliders ──────────────────────────────────────────────────────────────
cor_slider = widgets.IntSlider(
    value=n_cor // 2, min=0, max=n_cor - 1, step=1,
    description='Coronal Y:',
    style={'description_width': '90px'},
    layout=widgets.Layout(width='500px'),
    continuous_update=True
)
axl_slider = widgets.IntSlider(
    value=n_z // 2, min=0, max=n_z - 1, step=1,
    description='Axial Z:',
    style={'description_width': '90px'},
    layout=widgets.Layout(width='500px'),
    continuous_update=True
)
vmax_s = widgets.FloatSlider(
    value=SLICE_VMAX,
    min=max(1.0, float(vol[vol > 0].min())),
    max=float(vol.max()),
    step=max(1.0, SLICE_VMAX / 100),
    description='Color max:',
    style={'description_width': '90px'},
    layout=widgets.Layout(width='500px'),
    readout_format='.0f',
    continuous_update=False
)
vmax_label_s = widgets.Label(value=units)

out_s = widgets.Output()

def render_slices(y_idx, z_idx, vmax):
    cor_img = vol[y_idx, :, :]          # (Z, X) — coronal slice
    axl_img = vol[:, z_idx, :]          # (Y, X) — axial cross-section

    # Physical aspect ratios: data units are pixels; spacing converts to mm
    # aspect = (mm per y-data-unit) / (mm per x-data-unit)
    cor_aspect = z_spacing / x_spacing  # coronal: rows=Z, cols=X
    axl_aspect = y_spacing / x_spacing  # axial:   rows=Y (slice stack), cols=X

    with out_s:
        out_s.clear_output(wait=True)
        fig = plt.figure(figsize=(14, 5), facecolor='black')
        gs  = gridspec.GridSpec(1, 3, width_ratios=[1, 1, 0.05], figure=fig)
        ax_cor  = fig.add_subplot(gs[0])
        ax_axl  = fig.add_subplot(gs[1])
        ax_cbar = fig.add_subplot(gs[2])
        fig.patch.set_facecolor('black')

        kw = dict(cmap='hot', vmin=0, vmax=vmax, interpolation='none')

        # ── Coronal slice ─────────────────────────────────────────────────
        ax_cor.set_facecolor('black')
        im = ax_cor.imshow(np.flipud(cor_img), aspect=cor_aspect, **kw)
        y_mm = slice_list[y_idx][0]
        ax_cor.set_title(f'Coronal slice  Y={y_mm:.1f} mm  ({y_idx}/{n_cor-1})',
                         color='white', fontsize=11)
        ax_cor.set_xlabel('← Left   Right →  (X)', color='#888', fontsize=9)
        ax_cor.set_ylabel('Inferior ↑ Superior  (Z)', color='#888', fontsize=9)
        ax_cor.tick_params(colors='#555')
        for sp in ax_cor.spines.values(): sp.set_edgecolor('#333')

        # ── Axial cross-section ───────────────────────────────────────────
        ax_axl.set_facecolor('black')
        ax_axl.imshow(axl_img, aspect=axl_aspect, **kw)
        ax_axl.set_title(f'Axial cross-section  Z row {z_idx}/{n_z-1}\n'
                         f'(aspect = y_spacing/x_spacing = {axl_aspect:.2f})',
                         color='white', fontsize=11)
        ax_axl.set_xlabel('← Left   Right →  (X)', color='#888', fontsize=9)
        ax_axl.set_ylabel('Anterior ← → Posterior  (Y)', color='#888', fontsize=9)
        ax_axl.tick_params(colors='#555')
        for sp in ax_axl.spines.values(): sp.set_edgecolor('#333')

        # ── Shared colorbar on dedicated axis ─────────────────────────────
        ax_cbar.set_facecolor('black')
        cbar = fig.colorbar(im, cax=ax_cbar)
        cbar.set_label(units, color='white', fontsize=9)
        cbar.ax.yaxis.set_tick_params(color='white')
        plt.setp(cbar.ax.yaxis.get_ticklabels(), color='white', fontsize=8)

        fig.subplots_adjust(left=0.07, right=0.95, wspace=0.35)
        plt.show()

widgets.interactive_output(
    render_slices, {'y_idx': cor_slider, 'z_idx': axl_slider, 'vmax': vmax_s}
)
display(widgets.VBox([cor_slider, axl_slider,
                      widgets.HBox([vmax_s, vmax_label_s])]),
        out_s)
render_slices(n_cor // 2, n_z // 2, SLICE_VMAX)

## Step 4 — Pre-compute rotating MIPs (coronal geometry)
For a coronal volume `(Y, Z, X)`:
- **Resample** along Y so all three spacings are equal (isotropic in Y–X), avoiding stretching at oblique angles.
- **Rotate** in the `(Y, X)` plane — `axes=(0, 2)` — which rotates around the vertical **Z** axis.
- **Project** along Y (`axis=0`) to get a `(Z, X)` front-facing MIP.

~2–4 minutes once; the angle slider is instant after.

In [ ]:
from scipy.ndimage import rotate, zoom
import time

# ── Resample Y axis to match X pixel spacing (isotropic in Y-X plane) ───
y_zoom = y_spacing / x_spacing   # >1 if slices are spaced wider than pixels
if abs(y_zoom - 1.0) > 0.01:
    print(f'Resampling Y axis: zoom factor {y_zoom:.4f}  ({y_spacing:.3f} mm → {x_spacing:.3f} mm)')
    vol_iso = zoom(vol, (y_zoom, 1.0, 1.0), order=1)
else:
    vol_iso = vol
print(f'Isotropic volume shape: {vol_iso.shape}  (Y, Z, X)')

# ── Pre-compute rotating MIPs ────────────────────────────────────────────
ANGLE_STEP = 5
angles     = np.arange(0, 360, ANGLE_STEP)            # 72 projections
# Projection shape: (Z, X) = (vol_iso.shape[1], vol_iso.shape[2])
mip_stack  = np.zeros((len(angles), vol_iso.shape[1], vol_iso.shape[2]), dtype=np.float32)

t0 = time.time()
for i, angle in enumerate(angles):
    # Rotate in (Y, X) plane = around vertical Z axis
    rot          = rotate(vol_iso, angle, axes=(0, 2), reshape=False, order=1, prefilter=False)
    mip_stack[i] = np.max(rot, axis=0)   # project along Y → shape (Z, X)
    if (i + 1) % 10 == 0:
        print(f'  {i+1}/{len(angles)} angles  ({time.time()-t0:.0f}s elapsed)')

print(f'Done. {len(angles)} projections in {time.time()-t0:.0f}s.')
VMAX_DEFAULT = float(np.percentile(vol[vol > 0], 99))
print(f'Suggested color max (99th percentile): {VMAX_DEFAULT:.0f} {units}')

## Step 5 — Interactive rotating MIP viewer
- **Angle** — rotates the projection view around the vertical axis (0°–355°)
- **Color max** — sets the upper end of the hot colormap; lower = higher contrast on faint structures

In [ ]:
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# ── Sliders ──────────────────────────────────────────────────────────────
angle_slider = widgets.IntSlider(
    value=0, min=0, max=355, step=ANGLE_STEP,
    description='Angle (°):',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='500px'),
    continuous_update=True
)

vmax_slider = widgets.FloatSlider(
    value=VMAX_DEFAULT,
    min=max(1.0, float(vol[vol > 0].min())),
    max=float(vol.max()),
    step=max(1.0, VMAX_DEFAULT / 100),
    description='Color max:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='500px'),
    readout_format='.0f',
    continuous_update=False
)

vmax_label = widgets.Label(value=units)
controls   = widgets.VBox([
    widgets.HBox([angle_slider]),
    widgets.HBox([vmax_slider, vmax_label]),
])

out = widgets.Output()

# ── Render function ───────────────────────────────────────────────────────
def render(angle, vmax):
    idx = int(round(angle / ANGLE_STEP)) % len(angles)
    img = np.flipud(mip_stack[idx])   # Z axis: superior at top

    with out:
        out.clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(10, 5), facecolor='black')
        ax.set_facecolor('black')
        im = ax.imshow(img, cmap='hot', vmin=0, vmax=vmax,
                       aspect='equal', interpolation='none')

        ax.set_title(f'PET MIP  S46712  |  {angle}°  |  color max {vmax:.0f} {units}',
                     color='white', fontsize=12, pad=8)
        ax.set_xlabel('← Left   Right →', color='#888', fontsize=9)
        ax.set_ylabel('Inferior ↑ Superior', color='#888', fontsize=9)
        ax.tick_params(colors='#555')
        for sp in ax.spines.values():
            sp.set_edgecolor('#333')

        cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
        cbar.set_label(units, color='white', fontsize=9)
        cbar.ax.yaxis.set_tick_params(color='white')
        plt.setp(cbar.ax.yaxis.get_ticklabels(), color='white', fontsize=8)

        plt.tight_layout()
        plt.show()

# ── Wire up ───────────────────────────────────────────────────────────────
widgets.interactive_output(render, {'angle': angle_slider, 'vmax': vmax_slider})
display(controls, out)
render(0, VMAX_DEFAULT)   # draw immediately

## (Optional) Export rotating MIP as animated GIF
Saves one full 360° rotation at the current color-max setting.

In [ ]:
from PIL import Image as PILImage
import io

VMAX_GIF = vmax_slider.value
frames   = []

for i in range(len(angles)):
    img = np.flipud(mip_stack[i])
    fig, ax = plt.subplots(figsize=(10, 5), facecolor='black')
    ax.set_facecolor('black')
    ax.imshow(img, cmap='hot', vmin=0, vmax=VMAX_GIF,
              aspect='equal', interpolation='none')
    ax.set_title(f'PET MIP  S46712  {angles[i]:.0f}°', color='white', fontsize=10)
    ax.axis('off')
    plt.tight_layout(pad=0.3)

    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=100, facecolor='black')
    plt.close(fig)
    buf.seek(0)
    frames.append(PILImage.open(buf).convert('P', palette=PILImage.ADAPTIVE))

gif_path = '/content/PET_rotating_MIP_S46712.gif'
frames[0].save(
    gif_path, save_all=True, append_images=frames[1:],
    loop=0, duration=80   # 80 ms per frame ≈ 12 fps
)
print(f'Saved: {gif_path}')

from google.colab import files
files.download(gif_path)